In [0]:
storage_account_name = ""
storage_account_key = ""
spark.conf.set(
f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
storage_account_key
)

In [0]:
# Read gold layer
train_gold_path = f"abfss://curated@{storage_account_name}.dfs.core.windows.net/FD001/train_gold"
test_gold_path = f"abfss://curated@{storage_account_name}.dfs.core.windows.net/FD001/test_gold"

train_df = spark.read.parquet(train_gold_path)
test_df = spark.read.parquet(test_gold_path)

print("=== GOLD DATA LOADED ===")
print(f"Train rows : {train_df.count()}")
print(f"Test rows  : {test_df.count()}")
print(f"Columns    : {train_df.columns}")
display(train_df.limit(5))

=== GOLD DATA LOADED ===
Train rows : 20631
Test rows  : 13096
Columns    : ['engine_id', 'cycle', 'RUL', 'op_setting_1_scaled', 'op_setting_2_scaled', 'op_setting_3_scaled', 'sensor_2_scaled', 'sensor_3_scaled', 'sensor_4_scaled', 'sensor_7_scaled', 'sensor_8_scaled', 'sensor_9_scaled', 'sensor_11_scaled', 'sensor_12_scaled', 'sensor_13_scaled', 'sensor_14_scaled', 'sensor_15_scaled', 'sensor_17_scaled', 'sensor_20_scaled', 'sensor_21_scaled']


engine_id,cycle,RUL,op_setting_1_scaled,op_setting_2_scaled,op_setting_3_scaled,sensor_2_scaled,sensor_3_scaled,sensor_4_scaled,sensor_7_scaled,sensor_8_scaled,sensor_9_scaled,sensor_11_scaled,sensor_12_scaled,sensor_13_scaled,sensor_14_scaled,sensor_15_scaled,sensor_17_scaled,sensor_20_scaled,sensor_21_scaled
1,1,125,0.45977011694199915,0.16666669091985756,0.5,0.1837301222538836,0.4067999584849122,0.3097565599409842,0.7262469900240798,0.24260355029585798,0.10975513566570848,0.3690491868792603,0.6332556838146302,0.20603015075376885,0.19960893108670288,0.36398862489679845,0.3333333333333333,0.7131793261297706,0.7246622244462588
1,2,125,0.6091954026064614,0.25,0.5,0.28313264086772677,0.45301742835320824,0.35263366124452394,0.6280210329745933,0.21227810650887574,0.10024188459650844,0.3809542190998224,0.765463749821059,0.27961234745154345,0.16281519222095392,0.41131272360333915,0.3333333333333333,0.6666666666666667,0.7310121414838421
1,3,125,0.25287355829661745,0.75,0.5,0.34335876459233383,0.3695215386130235,0.3705259080062807,0.7101479188166495,0.2729289940828402,0.14004329383720115,0.2500011353263609,0.7953045899975273,0.22074659009332376,0.17179314671887638,0.35744610586184755,0.16666666666666666,0.6279075956778624,0.6213753325080987
1,4,125,0.5402298830580008,0.5,0.5,0.34335876459233383,0.25615873666394334,0.3311951105927541,0.7407440169050077,0.3184171597633136,0.12451798359391432,0.1666681804351479,0.889121692846267,0.29432878679109836,0.17488994887378703,0.16660489863315292,0.3333333333333333,0.5736443048680234,0.6623851036371777
1,5,125,0.39080459739353857,0.33333334545992876,0.5,0.34938873058185493,0.2574653974404352,0.404624825363594,0.6682785394859698,0.24260355029585798,0.14995968590058192,0.255952516110281,0.7462682682422144,0.2354630294328787,0.17473372289038228,0.4020805430694432,0.41666666666666663,0.5891485246890581,0.7045010403223683


In [0]:
# tsfresh needs features separate from labels
# So we split RUL into its own dataframe

# Training labels — one RUL per engine (at last cycle)
from pyspark.sql.functions import max as spark_max, col
from pyspark.sql import Window

# Get RUL at cycle 1 per engine = max RUL = total life
# This is what we predict for each engine
window = Window.partitionBy("engine_id")

train_labels = train_df.select("engine_id", "cycle", "RUL")

# Get one label per engine — RUL at the last observed cycle
train_labels_final = train_df \
    .groupBy("engine_id") \
    .agg(spark_max("cycle").alias("max_cycle")) \
    .join(
        train_labels,
        (train_labels.engine_id == train_df.engine_id) & 
        (train_labels.cycle == spark_max("cycle").over(window)),
        "left"
    )

train_labels_df = train_df.select("engine_id", "cycle", "RUL")

print("Labels separated")
display(train_labels_df.limit(10))

Labels separated


engine_id,cycle,RUL
1,1,125
1,2,125
1,3,125
1,4,125
1,5,125
1,6,125
1,7,125
1,8,125
1,9,125
1,10,125


In [0]:
# tsfresh needs: id column, time column, value columns
# Drop RUL from features dataframe

# Get scaled sensor and op_setting columns
feature_cols = [c for c in train_df.columns 
                if c.endswith("_scaled")]

print(f"Feature columns for tsfresh: {feature_cols}")
print(f"Total feature columns: {len(feature_cols)}")

# Build tsfresh-ready dataframes
# Must have: engine_id (id), cycle (time), + sensor columns
tsfresh_cols = ["engine_id", "cycle"] + feature_cols

train_tsfresh = train_df.select(tsfresh_cols)
test_tsfresh = test_df.select(tsfresh_cols)

print("\ntsfresh-ready format confirmed")
print(f"Train shape: {train_tsfresh.count()} rows x {len(train_tsfresh.columns)} cols")
print(f"Test shape : {test_tsfresh.count()} rows x {len(test_tsfresh.columns)} cols")
display(train_tsfresh.limit(5))

Feature columns for tsfresh: ['op_setting_1_scaled', 'op_setting_2_scaled', 'op_setting_3_scaled', 'sensor_2_scaled', 'sensor_3_scaled', 'sensor_4_scaled', 'sensor_7_scaled', 'sensor_8_scaled', 'sensor_9_scaled', 'sensor_11_scaled', 'sensor_12_scaled', 'sensor_13_scaled', 'sensor_14_scaled', 'sensor_15_scaled', 'sensor_17_scaled', 'sensor_20_scaled', 'sensor_21_scaled']
Total feature columns: 17

tsfresh-ready format confirmed
Train shape: 20631 rows x 19 cols
Test shape : 13096 rows x 19 cols


engine_id,cycle,op_setting_1_scaled,op_setting_2_scaled,op_setting_3_scaled,sensor_2_scaled,sensor_3_scaled,sensor_4_scaled,sensor_7_scaled,sensor_8_scaled,sensor_9_scaled,sensor_11_scaled,sensor_12_scaled,sensor_13_scaled,sensor_14_scaled,sensor_15_scaled,sensor_17_scaled,sensor_20_scaled,sensor_21_scaled
1,1,0.45977011694199915,0.16666669091985756,0.5,0.1837301222538836,0.4067999584849122,0.3097565599409842,0.7262469900240798,0.24260355029585798,0.10975513566570848,0.3690491868792603,0.6332556838146302,0.20603015075376885,0.19960893108670288,0.36398862489679845,0.3333333333333333,0.7131793261297706,0.7246622244462588
1,2,0.6091954026064614,0.25,0.5,0.28313264086772677,0.45301742835320824,0.35263366124452394,0.6280210329745933,0.21227810650887574,0.10024188459650844,0.3809542190998224,0.765463749821059,0.27961234745154345,0.16281519222095392,0.41131272360333915,0.3333333333333333,0.6666666666666667,0.7310121414838421
1,3,0.25287355829661745,0.75,0.5,0.34335876459233383,0.3695215386130235,0.3705259080062807,0.7101479188166495,0.2729289940828402,0.14004329383720115,0.2500011353263609,0.7953045899975273,0.22074659009332376,0.17179314671887638,0.35744610586184755,0.16666666666666666,0.6279075956778624,0.6213753325080987
1,4,0.5402298830580008,0.5,0.5,0.34335876459233383,0.25615873666394334,0.3311951105927541,0.7407440169050077,0.3184171597633136,0.12451798359391432,0.1666681804351479,0.889121692846267,0.29432878679109836,0.17488994887378703,0.16660489863315292,0.3333333333333333,0.5736443048680234,0.6623851036371777
1,5,0.39080459739353857,0.33333334545992876,0.5,0.34938873058185493,0.2574653974404352,0.404624825363594,0.6682785394859698,0.24260355029585798,0.14995968590058192,0.255952516110281,0.7462682682422144,0.2354630294328787,0.17473372289038228,0.4020805430694432,0.41666666666666663,0.5891485246890581,0.7045010403223683


In [0]:
from pyspark.sql.functions import min as spark_min

# For each engine we need ONE RUL label
# We take the RUL at the LAST cycle of each engine
# (which is 0 for training data since engines run to failure)
# But we need the RUL at the FIRST cycle = total useful life before clipping

# Actually for regression we want RUL at each cycle
# tsfresh will extract features per engine then we predict RUL per engine
# So we take RUL at the last observed cycle per engine

last_cycle_labels = train_df \
    .groupBy("engine_id") \
    .agg(spark_max("cycle").alias("last_cycle")) 

train_rul_labels = train_df.select("engine_id", "cycle", "RUL") \
                           .withColumnRenamed("RUL", "target_RUL")

print("=== RUL LABELS PER ENGINE PER CYCLE ===")
print(f"Total rows     : {train_rul_labels.count()}")
print(f"Total engines  : {train_rul_labels.select('engine_id').distinct().count()}")
train_rul_labels.describe("target_RUL").show()
display(train_rul_labels.limit(15))

=== RUL LABELS PER ENGINE PER CYCLE ===
Total rows     : 20631
Total engines  : 100
+-------+------------------+
|summary|        target_RUL|
+-------+------------------+
|  count|             20631|
|   mean| 86.82928602588338|
| stddev|41.673698854532724|
|    min|                 0|
|    max|               125|
+-------+------------------+



engine_id,cycle,target_RUL
1,1,125
1,2,125
1,3,125
1,4,125
1,5,125
1,6,125
1,7,125
1,8,125
1,9,125
1,10,125


In [0]:
# Write tsfresh-ready features
train_features_path = f"abfss://curated@{storage_account_name}.dfs.core.windows.net/FD001/train_tsfresh_ready"
test_features_path = f"abfss://curated@{storage_account_name}.dfs.core.windows.net/FD001/test_tsfresh_ready"
train_labels_path = f"abfss://curated@{storage_account_name}.dfs.core.windows.net/FD001/train_rul_labels"

train_tsfresh.write.mode("overwrite").parquet(train_features_path)
test_tsfresh.write.mode("overwrite").parquet(test_features_path)
train_rul_labels.write.mode("overwrite").parquet(train_labels_path)

print("tsfresh-ready data written to curated")
print(f"Train features : {train_features_path}")
print(f"Test features  : {test_features_path}")
print(f"Train labels   : {train_labels_path}")

tsfresh-ready data written to curated
Train features : abfss://curated@azuredatalake60304739.dfs.core.windows.net/FD001/train_tsfresh_ready
Test features  : abfss://curated@azuredatalake60304739.dfs.core.windows.net/FD001/test_tsfresh_ready
Train labels   : abfss://curated@azuredatalake60304739.dfs.core.windows.net/FD001/train_rul_labels


In [0]:
# Read back and verify
train_check = spark.read.parquet(train_features_path)
test_check = spark.read.parquet(test_features_path)
labels_check = spark.read.parquet(train_labels_path)

print("=== CURATED LAYER VERIFICATION ===")
print(f"Train features rows    : {train_check.count()}")
print(f"Test features rows     : {test_check.count()}")
print(f"Train labels (engines) : {labels_check.count()}")
print(f"Feature columns        : {len(train_check.columns)}")

print("\n=== FINAL SCHEMA ===")
train_check.printSchema()
display(train_check.limit(5))

print("\n=== LABELS SAMPLE ===")
display(labels_check.limit(10))

=== CURATED LAYER VERIFICATION ===
Train features rows    : 20631
Test features rows     : 13096
Train labels (engines) : 20631
Feature columns        : 19

=== FINAL SCHEMA ===
root
 |-- engine_id: integer (nullable = true)
 |-- cycle: integer (nullable = true)
 |-- op_setting_1_scaled: double (nullable = true)
 |-- op_setting_2_scaled: double (nullable = true)
 |-- op_setting_3_scaled: double (nullable = true)
 |-- sensor_2_scaled: double (nullable = true)
 |-- sensor_3_scaled: double (nullable = true)
 |-- sensor_4_scaled: double (nullable = true)
 |-- sensor_7_scaled: double (nullable = true)
 |-- sensor_8_scaled: double (nullable = true)
 |-- sensor_9_scaled: double (nullable = true)
 |-- sensor_11_scaled: double (nullable = true)
 |-- sensor_12_scaled: double (nullable = true)
 |-- sensor_13_scaled: double (nullable = true)
 |-- sensor_14_scaled: double (nullable = true)
 |-- sensor_15_scaled: double (nullable = true)
 |-- sensor_17_scaled: double (nullable = true)
 |-- sensor_20

engine_id,cycle,op_setting_1_scaled,op_setting_2_scaled,op_setting_3_scaled,sensor_2_scaled,sensor_3_scaled,sensor_4_scaled,sensor_7_scaled,sensor_8_scaled,sensor_9_scaled,sensor_11_scaled,sensor_12_scaled,sensor_13_scaled,sensor_14_scaled,sensor_15_scaled,sensor_17_scaled,sensor_20_scaled,sensor_21_scaled
1,1,0.45977011694199915,0.16666669091985756,0.5,0.1837301222538836,0.4067999584849122,0.3097565599409842,0.7262469900240798,0.24260355029585798,0.10975513566570848,0.3690491868792603,0.6332556838146302,0.20603015075376885,0.19960893108670288,0.36398862489679845,0.3333333333333333,0.7131793261297706,0.7246622244462588
1,2,0.6091954026064614,0.25,0.5,0.28313264086772677,0.45301742835320824,0.35263366124452394,0.6280210329745933,0.21227810650887574,0.10024188459650844,0.3809542190998224,0.765463749821059,0.27961234745154345,0.16281519222095392,0.41131272360333915,0.3333333333333333,0.6666666666666667,0.7310121414838421
1,3,0.25287355829661745,0.75,0.5,0.34335876459233383,0.3695215386130235,0.3705259080062807,0.7101479188166495,0.2729289940828402,0.14004329383720115,0.2500011353263609,0.7953045899975273,0.22074659009332376,0.17179314671887638,0.35744610586184755,0.16666666666666666,0.6279075956778624,0.6213753325080987
1,4,0.5402298830580008,0.5,0.5,0.34335876459233383,0.25615873666394334,0.3311951105927541,0.7407440169050077,0.3184171597633136,0.12451798359391432,0.1666681804351479,0.889121692846267,0.29432878679109836,0.17488994887378703,0.16660489863315292,0.3333333333333333,0.5736443048680234,0.6623851036371777
1,5,0.39080459739353857,0.33333334545992876,0.5,0.34938873058185493,0.2574653974404352,0.404624825363594,0.6682785394859698,0.24260355029585798,0.14995968590058192,0.255952516110281,0.7462682682422144,0.2354630294328787,0.17473372289038228,0.4020805430694432,0.41666666666666663,0.5891485246890581,0.7045010403223683



=== LABELS SAMPLE ===


engine_id,cycle,target_RUL
1,1,125
1,2,125
1,3,125
1,4,125
1,5,125
1,6,125
1,7,125
1,8,125
1,9,125
1,10,125
